# Table + Text RAG Assistant

## Unified QA Over Structured and Unstructured Data

**Project 27** - Advance RAG Learning Series

| Property | Value |
|----------|-------|
| Task | Combined tabular + text retrieval for question answering |
| Method | Intent routing + SQL-like table queries + dense text retrieval |
| Embeddings | sentence-transformers (all-MiniLM-L6-v2) |
| Data | Embedded CSV tables + text documents (company financials scenario) |
| Evaluation | Answer completeness, source type accuracy, retrieval quality |


## Project Overview

### The Problem: Questions That Span Tables and Text

Real-world knowledge bases mix **structured data** (CSV, databases, spreadsheets)
with **unstructured text** (reports, policies, meeting notes). A single question
might need both:

- *"Which company had the highest revenue growth AND what's their strategy?"*
  - Revenue growth is in the table. Strategy is in the text report.

- *"What was TechCorp's Q3 margin?"*
  - Pure table lookup.

- *"Why did DataFlow pivot to enterprise?"*
  - Pure text retrieval.

### How This Assistant Works

```
User question
      |
      v
Intent Router (classify: TABLE / TEXT / BOTH)
      |
      +---> TABLE: query pandas DataFrame
      +---> TEXT:  dense embedding search
      +---> BOTH:  table query + text retrieval
      |
      v
Merge evidence
      |
      v
Format answer with source attribution
```


## Learning Goals

1. Understand the difference between **structured** and **unstructured** retrieval
2. Build an intent router to classify question types
3. Implement table querying for structured data
4. Implement dense retrieval for unstructured text
5. Combine both retrieval types for hybrid questions
6. Evaluate retrieval quality per source type


## Problem Statement

Given a question about a company knowledge base that contains:
- **Financial tables** (revenue, margins, headcount, growth rates)
- **Text reports** (strategy notes, market analysis, product descriptions)

Build a system that:
1. Routes each question to the correct data source(s)
2. Extracts structured answers from tables (exact numbers)
3. Retrieves relevant text passages for context questions
4. Merges evidence when questions span both sources


## Why Table + Text RAG Matters

### Structured vs Unstructured Retrieval

| Aspect | Structured (Table) | Unstructured (Text) |
|--------|-------------------|--------------------|
| Data format | Rows, columns, typed values | Free-form paragraphs |
| Query method | Filter, aggregate, sort | Embedding similarity |
| Precision | Exact values (e.g. $42.1M) | Approximate relevance |
| Strength | Numerical comparisons | Explanations, context |
| Weakness | No context/reasoning | No exact numbers |
| Example | "Q3 revenue for TechCorp" | "Why did TechCorp pivot?" |

Neither approach alone handles mixed questions.
A combined system covers both precise facts and rich context.


## Environment Setup

In [ ]:
import subprocess, sys, warnings

def _install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["sentence-transformers"]:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        _install(pkg)

warnings.filterwarnings("ignore")
print("Environment ready.")


## Imports

In [ ]:
import re, random
from typing import List, Dict, Optional, Tuple
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

print("All imports loaded.")


## Configuration

In [ ]:
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
K_TEXT = 3       # text chunks to retrieve

print(f"Config: model={EMBEDDING_MODEL}, K_TEXT={K_TEXT}")


## Tabular Data: Company Financial Metrics

Structured data stored as a pandas DataFrame.
Each row is a company-quarter record with typed numeric columns.

This is the **structured** side of our knowledge base.
Questions about specific numbers, rankings, or comparisons
should be answered from this table.


In [ ]:
financials = pd.DataFrame([
    {"company": "TechCorp",   "quarter": "Q1-2025", "revenue_m": 42.1, "margin_pct": 18.3, "headcount": 850, "growth_pct": 12.5, "sector": "Enterprise SaaS"},
    {"company": "TechCorp",   "quarter": "Q2-2025", "revenue_m": 45.8, "margin_pct": 19.1, "headcount": 890, "growth_pct": 8.8,  "sector": "Enterprise SaaS"},
    {"company": "TechCorp",   "quarter": "Q3-2025", "revenue_m": 51.2, "margin_pct": 21.5, "headcount": 920, "growth_pct": 11.8, "sector": "Enterprise SaaS"},
    {"company": "TechCorp",   "quarter": "Q4-2025", "revenue_m": 55.0, "margin_pct": 22.0, "headcount": 950, "growth_pct": 7.4,  "sector": "Enterprise SaaS"},
    {"company": "DataFlow",   "quarter": "Q1-2025", "revenue_m": 28.4, "margin_pct": 14.2, "headcount": 420, "growth_pct": 22.0, "sector": "Data Analytics"},
    {"company": "DataFlow",   "quarter": "Q2-2025", "revenue_m": 33.6, "margin_pct": 15.8, "headcount": 480, "growth_pct": 18.3, "sector": "Data Analytics"},
    {"company": "DataFlow",   "quarter": "Q3-2025", "revenue_m": 38.1, "margin_pct": 16.4, "headcount": 520, "growth_pct": 13.4, "sector": "Data Analytics"},
    {"company": "DataFlow",   "quarter": "Q4-2025", "revenue_m": 41.5, "margin_pct": 17.0, "headcount": 560, "growth_pct": 8.9,  "sector": "Data Analytics"},
    {"company": "CloudNine",  "quarter": "Q1-2025", "revenue_m": 61.0, "margin_pct": 25.0, "headcount": 1200, "growth_pct": 5.2, "sector": "Cloud Infrastructure"},
    {"company": "CloudNine",  "quarter": "Q2-2025", "revenue_m": 63.5, "margin_pct": 26.1, "headcount": 1220, "growth_pct": 4.1, "sector": "Cloud Infrastructure"},
    {"company": "CloudNine",  "quarter": "Q3-2025", "revenue_m": 65.0, "margin_pct": 24.8, "headcount": 1180, "growth_pct": 2.4, "sector": "Cloud Infrastructure"},
    {"company": "CloudNine",  "quarter": "Q4-2025", "revenue_m": 64.2, "margin_pct": 23.5, "headcount": 1150, "growth_pct": -1.2, "sector": "Cloud Infrastructure"},
    {"company": "AIVentures", "quarter": "Q1-2025", "revenue_m": 15.0, "margin_pct": -5.0, "headcount": 200, "growth_pct": 45.0, "sector": "AI/ML Platform"},
    {"company": "AIVentures", "quarter": "Q2-2025", "revenue_m": 19.5, "margin_pct": -2.0, "headcount": 260, "growth_pct": 30.0, "sector": "AI/ML Platform"},
    {"company": "AIVentures", "quarter": "Q3-2025", "revenue_m": 24.0, "margin_pct": 3.0,  "headcount": 310, "growth_pct": 23.1, "sector": "AI/ML Platform"},
    {"company": "AIVentures", "quarter": "Q4-2025", "revenue_m": 29.8, "margin_pct": 8.5,  "headcount": 370, "growth_pct": 24.2, "sector": "AI/ML Platform"},
])

print(f"Financial table: {len(financials)} rows x {len(financials.columns)} columns")
print(f"Columns: {list(financials.columns)}")
print(f"Companies: {sorted(financials.company.unique())}")
financials.head(8)


## Text Documents: Company Reports and Analysis

Unstructured text documents providing context, strategy,
and qualitative insights. Questions about *why* or *how*
should be answered from these documents.


In [ ]:
text_docs = [
    {"id": "T01", "company": "TechCorp", "type": "strategy",
     "text": "TechCorp announced a strategic pivot to enterprise SaaS in early 2025, moving away from consumer products. CEO Sarah Chen stated the enterprise market offers higher margins and more predictable revenue streams. The company invested $15M in sales team expansion and enterprise-grade security certifications."},
    {"id": "T02", "company": "TechCorp", "type": "product",
     "text": "TechCorp launched its flagship enterprise product, TechSuite Pro, in Q2 2025. The platform integrates project management, communication, and analytics into a single dashboard. Early enterprise customers report 30% productivity gains. The product targets companies with 500+ employees."},
    {"id": "T03", "company": "TechCorp", "type": "market",
     "text": "TechCorp faces competition from established players like Salesforce and ServiceNow in the enterprise SaaS space. However, analysts note that TechCorp differentiates through AI-powered workflow automation and competitive pricing at 40% below market leaders. The company secured 12 Fortune 500 contracts in H2 2025."},
    {"id": "T04", "company": "DataFlow", "type": "strategy",
     "text": "DataFlow pivoted from self-service analytics to enterprise data platform in 2025. CTO Michael Park explained the shift was driven by demand for real-time streaming analytics and data governance features. The company acquired StreamPipe for $8M to bolster its real-time processing capabilities."},
    {"id": "T05", "company": "DataFlow", "type": "product",
     "text": "DataFlow released DataFlow Enterprise 3.0, featuring a no-code data pipeline builder. The platform supports over 200 data connectors and includes built-in anomaly detection using statistical models. Key differentiator: unified batch and streaming processing in one interface."},
    {"id": "T06", "company": "DataFlow", "type": "market",
     "text": "DataFlow competes with Databricks and Snowflake in the data platform market. While smaller in scale, DataFlow targets mid-market companies ($50M-$500M revenue) underserved by enterprise-first competitors. Customer retention rate stands at 94%, above industry average of 88%."},
    {"id": "T07", "company": "CloudNine", "type": "strategy",
     "text": "CloudNine adopted a cost-optimization strategy in 2025, reducing headcount by 5% while investing in automation. The company focused on margin preservation amid slowing cloud infrastructure growth. CloudNine launched a reserved capacity pricing model offering 40% discounts for annual commitments."},
    {"id": "T08", "company": "CloudNine", "type": "product",
     "text": "CloudNine introduced CloudNine Edge, a distributed cloud platform for latency-sensitive workloads. The product deploys compute nodes at 50+ edge locations globally. Target use cases include IoT data processing, real-time gaming backends, and content delivery networks."},
    {"id": "T09", "company": "CloudNine", "type": "market",
     "text": "CloudNine holds 8% of the cloud infrastructure market, behind AWS (32%), Azure (23%), and GCP (11%). Growth has slowed as the cloud market matures. Analysts suggest CloudNine needs to differentiate through specialized vertical solutions rather than competing on general infrastructure."},
    {"id": "T10", "company": "AIVentures", "type": "strategy",
     "text": "AIVentures is a venture-backed AI startup pursuing aggressive growth over profitability. The company raised a $50M Series B in Q1 2025 led by Sequoia Capital. The strategy focuses on becoming the default MLOps platform for mid-size engineering teams, targeting 10x user growth by 2027."},
    {"id": "T11", "company": "AIVentures", "type": "product",
     "text": "AIVentures offers AutoML Studio, an end-to-end machine learning platform. Features include automated feature engineering, model selection, hyperparameter tuning, and one-click deployment. The platform reached profitability per-unit in Q3 2025 when margins turned positive."},
    {"id": "T12", "company": "AIVentures", "type": "market",
     "text": "AIVentures competes with DataRobot, H2O.ai, and Google Vertex AI in the AutoML space. Despite being the smallest player, AIVentures has the fastest-growing user community with 15,000+ active developers. Its open-source SDK has over 8,000 GitHub stars."},
    {"id": "T13", "company": "General", "type": "industry",
     "text": "The enterprise software market grew 14% in 2025, reaching $350B globally. Key trends include AI integration across all product categories, shift from perpetual licenses to consumption-based pricing, and increased focus on data security and compliance. SaaS companies with AI features commanded 25% higher valuations."},
    {"id": "T14", "company": "General", "type": "industry",
     "text": "Venture capital investment in AI companies reached $80B in 2025, a 35% increase year-over-year. Enterprise AI platforms received the most funding, followed by vertical AI applications in healthcare and finance. Investors increasingly focus on path-to-profitability metrics over pure growth."},
]

print(f"Text documents: {len(text_docs)}")
for doc in text_docs:
    print(f"  [{doc['id']}] {doc['company']:>11s} / {doc['type']:<10s} | {doc['text'][:55]}...")


## Text Retriever: Dense Embedding Search

For unstructured text, we use dense embeddings to find
semantically relevant documents. This is standard RAG retrieval.


In [ ]:
print(f"Loading embedding model: {EMBEDDING_MODEL}...")
encoder = SentenceTransformer(EMBEDDING_MODEL)

doc_texts = [doc["text"] for doc in text_docs]
doc_embeddings = encoder.encode(doc_texts, convert_to_numpy=True, show_progress_bar=False)
doc_embeddings = doc_embeddings / np.linalg.norm(doc_embeddings, axis=1, keepdims=True)


def retrieve_text(query: str, k: int = K_TEXT) -> List[Dict]:
    """Retrieve top-k text documents by cosine similarity."""
    q_emb = encoder.encode([query], convert_to_numpy=True)
    q_emb = q_emb / np.linalg.norm(q_emb, axis=1, keepdims=True)
    scores = (doc_embeddings @ q_emb.T).flatten()
    top_idx = np.argsort(scores)[::-1][:k]
    return [{"id": text_docs[i]["id"], "company": text_docs[i]["company"],
             "type": text_docs[i]["type"], "text": text_docs[i]["text"],
             "score": float(scores[i])}
            for i in top_idx]


# Quick test
test = retrieve_text("Why did DataFlow pivot?", k=2)
print(f"\nRetriever ready (index: {doc_embeddings.shape})")
for r in test:
    print(f"  [{r["id"]}] {r["company"]} | score={r["score"]:.3f} | {r["text"][:55]}...")


## Table Query Engine: Structured Data Lookup

For structured data, we don't use embeddings. Instead, we
parse the question to identify:
- **Company name** (filter rows)
- **Quarter** (filter rows)
- **Metric** (select column)
- **Operation** (aggregate: max, min, avg, compare)

This is fundamentally different from text retrieval:
structured queries return **exact values**, not approximate matches.


In [ ]:
# Column aliases for natural language mapping
METRIC_MAP = {
    "revenue": "revenue_m",
    "revenues": "revenue_m",
    "sales": "revenue_m",
    "margin": "margin_pct",
    "margins": "margin_pct",
    "profit margin": "margin_pct",
    "headcount": "headcount",
    "employees": "headcount",
    "team size": "headcount",
    "staff": "headcount",
    "growth": "growth_pct",
    "growth rate": "growth_pct",
    "sector": "sector",
}

COMPANY_ALIASES = {
    "techcorp": "TechCorp",
    "tech corp": "TechCorp",
    "dataflow": "DataFlow",
    "data flow": "DataFlow",
    "cloudnine": "CloudNine",
    "cloud nine": "CloudNine",
    "aiventures": "AIVentures",
    "ai ventures": "AIVentures",
}

QUARTER_PATTERN = re.compile(r"Q[1-4][-\s]?2025", re.IGNORECASE)


def parse_table_query(question: str) -> Dict:
    """Parse a natural language question into table query components."""
    q_lower = question.lower()
    result = {"companies": [], "quarters": [], "metrics": [], "operation": None}
    
    # Extract companies
    for alias, name in COMPANY_ALIASES.items():
        if alias in q_lower:
            if name not in result["companies"]:
                result["companies"].append(name)
    
    # Extract quarters
    for m in QUARTER_PATTERN.finditer(question):
        q_str = m.group().upper().replace(" ", "-")
        if not q_str[2] == "-":
            q_str = q_str[:2] + "-" + q_str[2:]
        if q_str not in result["quarters"]:
            result["quarters"].append(q_str)
    
    # Extract metrics
    for phrase, col in METRIC_MAP.items():
        if phrase in q_lower:
            if col not in result["metrics"]:
                result["metrics"].append(col)
    
    # Detect operations
    if any(w in q_lower for w in ["highest", "most", "maximum", "max", "top", "best", "largest"]):
        result["operation"] = "max"
    elif any(w in q_lower for w in ["lowest", "least", "minimum", "min", "worst", "smallest"]):
        result["operation"] = "min"
    elif any(w in q_lower for w in ["average", "avg", "mean"]):
        result["operation"] = "avg"
    elif any(w in q_lower for w in ["total", "sum"]):
        result["operation"] = "sum"
    elif any(w in q_lower for w in ["compare", "difference", "vs", "versus"]):
        result["operation"] = "compare"
    
    return result


def query_table(question: str) -> Dict:
    """Execute a structured query against the financial table."""
    parsed = parse_table_query(question)
    df = financials.copy()
    
    # Apply company filter
    if parsed["companies"]:
        df = df[df["company"].isin(parsed["companies"])]
    
    # Apply quarter filter
    if parsed["quarters"]:
        df = df[df["quarter"].isin(parsed["quarters"])]
    
    if df.empty:
        return {"data": None, "parsed": parsed, "answer": "No matching records found."}
    
    # Select metrics
    metric_cols = parsed["metrics"] if parsed["metrics"] else ["revenue_m", "margin_pct", "growth_pct"]
    display_cols = ["company", "quarter"] + [c for c in metric_cols if c in df.columns]
    result_df = df[display_cols]
    
    # Apply operation
    op = parsed["operation"]
    answer_parts = []
    
    if op == "max" and metric_cols:
        col = metric_cols[0]
        if col in df.columns:
            idx = df[col].idxmax()
            row = df.loc[idx]
            answer_parts.append(f"Highest {col}: {row['company']} in {row['quarter']} with {row[col]}")
    elif op == "min" and metric_cols:
        col = metric_cols[0]
        if col in df.columns:
            idx = df[col].idxmin()
            row = df.loc[idx]
            answer_parts.append(f"Lowest {col}: {row['company']} in {row['quarter']} with {row[col]}")
    elif op == "avg" and metric_cols:
        for col in metric_cols:
            if col in df.columns:
                avg_val = df[col].mean()
                answer_parts.append(f"Average {col}: {avg_val:.1f}")
    elif op == "sum" and metric_cols:
        for col in metric_cols:
            if col in df.columns:
                total = df[col].sum()
                answer_parts.append(f"Total {col}: {total:.1f}")
    elif op == "compare" and len(parsed["companies"]) >= 2:
        for col in metric_cols:
            if col in df.columns:
                for comp in parsed["companies"]:
                    val = df[df["company"] == comp][col].mean()
                    answer_parts.append(f"{comp} avg {col}: {val:.1f}")
    
    if not answer_parts:
        answer_parts.append(result_df.to_string(index=False))
    
    return {
        "data": result_df,
        "parsed": parsed,
        "answer": " | ".join(answer_parts),
    }


# Demo
demo_q = "What was TechCorp revenue in Q3-2025?"
demo_r = query_table(demo_q)
print(f"Query: {demo_q}")
print(f"Parsed: {demo_r['parsed']}")
print(f"Answer: {demo_r['answer']}")


## Intent Router

The router classifies each question as:
- **TABLE**: needs numerical data from the DataFrame
- **TEXT**: needs qualitative/contextual information
- **BOTH**: needs numbers AND context

In production, an LLM handles intent classification.
Here we use keyword-based routing that captures common patterns.


In [ ]:
TABLE_SIGNALS = [
    "revenue", "margin", "headcount", "growth", "how much",
    "how many", "employees", "highest", "lowest", "compare",
    "total", "average", "q1", "q2", "q3", "q4", "number",
    "percentage", "profit", "sales", "cost", "budget",
]

TEXT_SIGNALS = [
    "why", "strategy", "explain", "describe", "how does",
    "what is their approach", "market", "competition", "product",
    "launched", "pivot", "vision", "plan", "trend", "industry",
    "competitor", "differentiate", "advantage", "feature",
]


def route_question(question: str) -> str:
    """Classify question intent: TABLE, TEXT, or BOTH."""
    q_lower = question.lower()
    
    table_score = sum(1 for s in TABLE_SIGNALS if s in q_lower)
    text_score = sum(1 for s in TEXT_SIGNALS if s in q_lower)
    
    if table_score > 0 and text_score > 0:
        return "BOTH"
    elif table_score > 0:
        return "TABLE"
    elif text_score > 0:
        return "TEXT"
    else:
        return "TEXT"  # default to text retrieval


# Demo
demo_routes = [
    "What was TechCorp revenue in Q3?",
    "Why did DataFlow pivot to enterprise?",
    "Which company had the highest growth and what is their strategy?",
]
print("Intent Routing Demo")
for q in demo_routes:
    print(f"  [{route_question(q):>5s}] {q}")


## Unified QA Pipeline

The pipeline routes each question, executes the appropriate
retrieval strategy, and returns evidence with source attribution.


In [ ]:
@dataclass
class QAResult:
    """Result from the Table+Text QA pipeline."""
    question: str
    intent: str               # TABLE, TEXT, BOTH
    table_answer: Optional[str]
    text_evidence: List[Dict]
    merged_answer: str
    sources_used: List[str]   # which source types provided evidence


def answer_question(question: str) -> QAResult:
    """Route question and retrieve evidence from appropriate sources."""
    intent = route_question(question)
    table_answer = None
    text_evidence = []
    sources_used = []
    answer_parts = []
    
    if intent in ("TABLE", "BOTH"):
        tq = query_table(question)
        table_answer = tq["answer"]
        sources_used.append("TABLE")
        answer_parts.append(f"[TABLE] {table_answer}")
    
    if intent in ("TEXT", "BOTH"):
        text_evidence = retrieve_text(question, k=K_TEXT)
        sources_used.append("TEXT")
        for doc in text_evidence:
            answer_parts.append(f"[TEXT {doc['id']}] {doc['text'][:120]}...")
    
    return QAResult(
        question=question,
        intent=intent,
        table_answer=table_answer,
        text_evidence=text_evidence,
        merged_answer="\n".join(answer_parts),
        sources_used=sources_used,
    )


print("Unified QA pipeline defined.")


## Test Questions

Questions covering all three intent types: TABLE, TEXT, BOTH.
Each has ground-truth expected intent and key evidence sources.


In [ ]:
test_questions = [
    # Pure TABLE questions
    {
        "question": "What was TechCorp revenue in Q3-2025?",
        "expected_intent": "TABLE",
        "expected_sources": ["TABLE"],
        "key_facts": ["51.2"],
        "category": "single-lookup",
    },
    {
        "question": "Which company had the highest revenue growth in Q1-2025?",
        "expected_intent": "TABLE",
        "expected_sources": ["TABLE"],
        "key_facts": ["AIVentures", "45.0"],
        "category": "aggregation",
    },
    {
        "question": "Compare TechCorp and DataFlow average margins",
        "expected_intent": "TABLE",
        "expected_sources": ["TABLE"],
        "key_facts": ["TechCorp", "DataFlow"],
        "category": "comparison",
    },
    {
        "question": "How many employees does CloudNine have in Q4-2025?",
        "expected_intent": "TABLE",
        "expected_sources": ["TABLE"],
        "key_facts": ["1150"],
        "category": "single-lookup",
    },
    # Pure TEXT questions
    {
        "question": "Why did DataFlow pivot to enterprise?",
        "expected_intent": "TEXT",
        "expected_sources": ["TEXT"],
        "key_facts": ["T04"],
        "category": "strategy",
    },
    {
        "question": "Describe TechCorp product features and competitive advantage",
        "expected_intent": "TEXT",
        "expected_sources": ["TEXT"],
        "key_facts": ["T02", "T03"],
        "category": "product-market",
    },
    {
        "question": "What industry trends shaped the market in 2025?",
        "expected_intent": "TEXT",
        "expected_sources": ["TEXT"],
        "key_facts": ["T13"],
        "category": "industry",
    },
    # BOTH questions (need table + text)
    {
        "question": "Which company had the highest growth and what is their strategy?",
        "expected_intent": "BOTH",
        "expected_sources": ["TABLE", "TEXT"],
        "key_facts": ["AIVentures", "T10"],
        "category": "hybrid-growth",
    },
    {
        "question": "What was CloudNine Q4-2025 revenue and why did growth slow?",
        "expected_intent": "BOTH",
        "expected_sources": ["TABLE", "TEXT"],
        "key_facts": ["64.2", "T07", "T09"],
        "category": "hybrid-explain",
    },
    {
        "question": "Compare TechCorp and DataFlow margins and explain their product strategy differences",
        "expected_intent": "BOTH",
        "expected_sources": ["TABLE", "TEXT"],
        "key_facts": ["TechCorp", "DataFlow", "T01", "T04"],
        "category": "hybrid-compare",
    },
]

print(f"Test questions: {len(test_questions)}")
for i, q in enumerate(test_questions, 1):
    print(f"  Q{i} [{q["expected_intent"]:>5s}] {q["category"]:>16s} | {q["question"][:55]}...")


## Run Benchmark

In [ ]:
all_results = []

for tq in test_questions:
    result = answer_question(tq["question"])
    all_results.append({"tq": tq, "result": result})

print(f"Benchmark complete: {len(all_results)} questions answered.")


## Per-Question Results

In [ ]:
for i, entry in enumerate(all_results, 1):
    tq = entry["tq"]
    r = entry["result"]
    intent_match = r.intent == tq["expected_intent"]
    
    print(f"\nQ{i}: {tq['question']}")
    print(f"  Expected: {tq['expected_intent']}  |  Predicted: {r.intent}  |  {"CORRECT" if intent_match else "WRONG"}")
    print(f"  Sources used: {r.sources_used}")
    
    if r.table_answer:
        print(f"  Table answer: {r.table_answer[:100]}")
    
    if r.text_evidence:
        for doc in r.text_evidence[:2]:
            print(f"  Text [{doc["id"]}] ({doc["company"]}/{doc["type"]}) score={doc["score"]:.3f}")
    
    # Check key facts
    answer_text = r.merged_answer
    facts_found = [f for f in tq["key_facts"] if f in answer_text]
    facts_missed = [f for f in tq["key_facts"] if f not in answer_text]
    print(f"  Key facts found: {facts_found}  |  Missed: {facts_missed}")


## Aggregate Metrics

In [ ]:
# Intent routing accuracy
intent_correct = sum(1 for e in all_results if e["result"].intent == e["tq"]["expected_intent"])
intent_accuracy = intent_correct / len(all_results)

# Source type accuracy
source_correct = sum(
    1 for e in all_results
    if set(e["result"].sources_used) == set(e["tq"]["expected_sources"])
)
source_accuracy = source_correct / len(all_results)

# Key facts recall
total_facts = sum(len(e["tq"]["key_facts"]) for e in all_results)
found_facts = sum(
    sum(1 for f in e["tq"]["key_facts"] if f in e["result"].merged_answer)
    for e in all_results
)
fact_recall = found_facts / total_facts if total_facts > 0 else 0

print("Aggregate Metrics")
print("=" * 40)
print(f"  Intent routing accuracy:  {intent_accuracy:.1%} ({intent_correct}/{len(all_results)})")
print(f"  Source type accuracy:     {source_accuracy:.1%} ({source_correct}/{len(all_results)})")
print(f"  Key facts recall:         {fact_recall:.1%} ({found_facts}/{total_facts})")

# Per intent type
print("\nBy intent type:")
for itype in ["TABLE", "TEXT", "BOTH"]:
    subset = [e for e in all_results if e["tq"]["expected_intent"] == itype]
    if not subset:
        continue
    correct = sum(1 for e in subset if e["result"].intent == e["tq"]["expected_intent"])
    total_f = sum(len(e["tq"]["key_facts"]) for e in subset)
    found_f = sum(sum(1 for f in e["tq"]["key_facts"] if f in e["result"].merged_answer) for e in subset)
    print(f"  {itype:>5s}: routing={correct}/{len(subset)}, facts={found_f}/{total_f}")


## How Structured vs Unstructured Retrieval Differ

### Structured Retrieval (Tables)

```
Question: "TechCorp Q3 revenue"
    |
    v
Parse: company=TechCorp, quarter=Q3-2025, metric=revenue_m
    |
    v
Filter: df[(company == TechCorp) & (quarter == Q3-2025)]
    |
    v
Result: 51.2 (exact)
```

- **Deterministic**: same question always gets same answer
- **Exact**: returns typed values (numbers, dates, categories)
- **Requires parsing**: must understand schema and map natural language to columns
- **Fails on ambiguity**: "How well is TechCorp doing?" is too vague for table lookup

### Unstructured Retrieval (Text)

```
Question: "Why did DataFlow pivot?"
    |
    v
Encode question into embedding vector
    |
    v
Cosine similarity against all document embeddings
    |
    v
Result: T04 (score=0.72) - strategy document about DataFlow pivot
```

- **Probabilistic**: returns ranked documents by relevance score
- **Approximate**: may retrieve partially relevant documents
- **No parsing needed**: works on any free-text question
- **Handles ambiguity**: semantic similarity captures fuzzy intent

### When to Use Each

| Question type | Best retrieval | Why |
|--------------|---------------|-----|
| Exact number lookup | Table | Precise value from a cell |
| Aggregation (max, avg) | Table | Computed from multiple rows |
| "Why" / "How" questions | Text | Need narrative explanation |
| Strategy / market context | Text | Need qualitative analysis |
| Number + explanation | Both | Table for fact, text for context |


## Qualitative Comparison: Three Question Types

In [ ]:
examples = [
    ("Pure TABLE", "What was TechCorp revenue in Q3-2025?"),
    ("Pure TEXT", "Why did DataFlow pivot to enterprise?"),
    ("BOTH needed", "Which company had the highest growth and what is their strategy?"),
]

for label, question in examples:
    result = answer_question(question)
    print(f"\n{"=" * 60}")
    print(f"Example: {label}")
    print(f"Q: {question}")
    print(f"Intent: {result.intent} | Sources: {result.sources_used}")
    print(f"\nAnswer:")
    print(result.merged_answer[:300])


## Error Analysis

Where does the system fail, and why?


In [ ]:
print("Error Analysis")
print("=" * 50)

# Routing errors
routing_errors = [(i+1, e) for i, e in enumerate(all_results)
                  if e["result"].intent != e["tq"]["expected_intent"]]
print(f"\nRouting errors: {len(routing_errors)}/{len(all_results)}")
for idx, e in routing_errors:
    print(f"  Q{idx}: expected={e['tq']['expected_intent']} got={e['result'].intent}")
    print(f"    {e['tq']['question'][:70]}")

# Fact misses
print(f"\nFact coverage errors:")
for i, e in enumerate(all_results, 1):
    missed = [f for f in e["tq"]["key_facts"] if f not in e["result"].merged_answer]
    if missed:
        print(f"  Q{i}: missed {missed}")
        print(f"    {e["tq"]["question"][:70]}")
        print(f"    Intent: {e["result"].intent}, Sources: {e["result"].sources_used}")
        if e["result"].table_answer:
            print(f"    Table answer: {e["result"].table_answer[:80]}")
        if e["result"].text_evidence:
            print(f"    Text retrieved: {[d["id"] for d in e["result"].text_evidence]}")

# Root causes
print("\nCommon failure modes:")
print("  1. Routing ambiguity: questions with both number and context keywords")
print("  2. Table parse failures: metric not recognized or wrong column mapped")
print("  3. Text retrieval misses: relevant doc ranked below K threshold")
print("  4. BOTH underserve: table answer present but text retrieval misses context doc")


## Limitations

1. **Rule-based routing.** Keyword matching misclassifies edge cases.
   An LLM router would handle subtle intent better.

2. **No SQL generation.** The table query engine uses pattern matching,
   not text-to-SQL. Complex joins, subqueries, and conditional
   aggregations are not supported.

3. **No answer generation.** We return raw evidence, not a
   synthesized natural language answer.

4. **Small corpus.** 14 text documents and 16 financial records.
   Real systems have millions of both.

5. **Single table.** Real scenarios have multiple related tables
   requiring joins and foreign key navigation.

6. **No schema discovery.** The metric map is hardcoded.
   A production system would infer column semantics dynamically.


## Common Mistakes

| Mistake | Why it fails | Fix |
|---------|-------------|-----|
| Embedding tabular data as text | Loses numerical precision; "42.1M" is not semantically similar to "$42 million" | Use structured query engine for tables |
| Using text retrieval for aggregation | Embedding search can't compute MAX, AVG, SUM | Route aggregation queries to table engine |
| No intent routing | All questions go through same retrieval path | Classify intent first, then route |
| Hardcoded column names | Breaks when schema changes | Use schema-aware mapping or text-to-SQL |
| Ignoring mixed questions | BOTH questions answered partially | Detect and dispatch to both engines |
| Over-engineering routing | Complex NLU for simple keyword patterns | Start simple, upgrade to LLM when needed |


## Mini Challenge

1. **Text-to-SQL.** Replace the pattern-matching table engine with
   an LLM that generates pandas queries from natural language.

2. **Multiple tables.** Add a second table (e.g., product metrics)
   and implement cross-table queries with join logic.

3. **LLM router.** Replace keyword routing with an embedding-based
   classifier trained on intent examples.

4. **Answer synthesis.** Add an LLM that takes table data + text
   evidence and generates a coherent natural language answer.

5. **Confidence scoring.** Assign confidence to each answer based on
   table match quality and text retrieval scores.


## Production Considerations

| Aspect | Approach |
|--------|----------|
| **Intent classification** | Fine-tuned classifier or LLM for routing |
| **Table querying** | Text-to-SQL (e.g., DIN-SQL, DAIL-SQL) for complex queries |
| **Schema management** | Auto-discover column types and semantics |
| **Multiple tables** | Support JOINs, foreign keys, multi-table queries |
| **Scaling text** | Vector DB (FAISS, Milvus) for large document collections |
| **Caching** | Cache table queries and frequent text retrievals |
| **Monitoring** | Track routing accuracy, answer quality, latency per source |
| **Security** | Sanitize table queries to prevent injection attacks |


## How to Improve This Project

1. **Text-to-SQL** -- Replace pattern matching with learned SQL generation.
2. **LLM routing** -- Use a small classifier or LLM for intent detection.
3. **Cross-encoder reranking** -- Rerank text results after initial retrieval.
4. **Answer generation** -- Add LLM to synthesize final answers.
5. **Multi-table support** -- Handle JOINs across related tables.
6. **Schema inference** -- Auto-detect column semantics from data.
7. **Feedback loop** -- Track user satisfaction to improve routing.


## Key Takeaways

1. **Structured and unstructured retrieval are fundamentally different.**
   Tables need exact queries (filter, aggregate). Text needs semantic
   similarity search. They are complementary, not interchangeable.

2. **Intent routing is critical.** Wrong routing sends number questions
   to text search (bad precision) or context questions to tables (no results).

3. **BOTH questions are the hardest.** They need evidence from two
   different retrieval systems, merged coherently.

4. **Don't embed tables as text.** Embedding "revenue: 42.1M" destroys
   numerical precision. Use a proper query engine for structured data.

5. **Start simple, upgrade gradually.** Keyword routing works for
   clear-intent questions. Add LLM routing when edge cases matter.

6. **Source attribution matters.** Users trust answers more when they
   can see whether the data came from a table or a document.

7. **This pattern scales.** The same architecture (router + specialized
   engines) applies to SQL databases, APIs, knowledge graphs,
   and any mixed-source knowledge base.
